# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Imports

In [1]:
from _imports import *
from araras.ml.optuna.model_tools import plot_model_param_distribution

2025-07-25 13:31:32.744147: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-25 13:31:32.758198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753461092.775322  227620 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753461092.780211  227620 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-25 13:31:32.797183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.2. Policy

In [2]:
POLICY = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(POLICY)

### 1.3. Constants

In [3]:
DATA_SEED = 99
TRAIN_SEED = 111

## 2. Data Loading and Preprocessing

In [4]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [5]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 3. Hyperparameters

In [6]:
kparams = KParams.default()
kparams.learning_rate = 7e-5

I0000 00:00:1753461096.034282  227620 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5193 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 4. Model Architectures

In [7]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 4)

    for i in range(num_conv_layers):
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(3, 9),
            kernel_size_step=2,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        #! pool size = 1 means no downsampling
        pool_size = trial.suggest_int(f"pool_size_{i}", 1, 4, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    cnn_out = x  # keep 3D shape here

    # ---------------------- Dense branch decisions --------------------------- #
    use_cnn_as_dense = trial.suggest_categorical("use_cnn_as_dense", [True, False])
    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 3)
    pooling_type = trial.suggest_categorical("pooling_type", ["flatten", "max", "average"])

    if use_cnn_as_dense:
        x = cnn_out  # still 3D
        for i in range(num_dense_layers):
            x = build_dense_as_conv1d(
                trial=trial,
                kparams=kparams,
                x=x,
                name_prefix=f"cnn_dense_{i}",
                filters_range=(250, 600),
                filters_step=50,
                kernel_initializer=initializer,
            )
        # collapse after the conv-as-dense stack
        if pooling_type == "flatten":
            x = layers.Flatten(name="flatten_after_cnn_dense")(x)
        elif pooling_type == "max":
            x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)
        else:
            x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(x)
    else:
        # collapse first
        if pooling_type == "flatten":
            x = layers.Flatten(name="flatten_cnn_output")(cnn_out)
        elif pooling_type == "max":
            x = layers.GlobalMaxPooling1D(name="global_max_pooling")(cnn_out)
        else:
            x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(cnn_out)

        for i in range(num_dense_layers):
            x = build_dnn(
                trial=trial,
                kparams=kparams,
                x=x,
                name_prefix=f"dense_{i}",
                units_range=(250, 600),
                units_step=50,
                dropout_rate_range=(0.0, 0.4),
                dropout_rate_step=0.2,
                kernel_initializer=initializer,
            )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,
    )

    return model

## 5. Optuna Ask

In [8]:
base_path = "runs/nas_cnn1d_v0.0_model_param_distribution/"

plot_model_param_distribution(
    lambda trial: build_model(trial=trial, kparams=kparams, show_summary=False),
    bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
    batch_size=64,
    n_trials=3000,
    fig_save_path=f"{base_path}model_param_distribution.png",
    csv_path=f"{base_path}model_param_distribution.csv",
    logs_dir=f"{base_path}logs/",
    figsize=(18, 6),
)

[I 2025-07-25 13:31:36,156] A new study created in memory with name: no-name-6cb14bf8-4b7b-4726-ac44-b56b8f86d68c
  0% [...............................] 0/3000 in ?2025-07-25 13:31:36.811484: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
2025-07-25 13:31:36.835660: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
I0000 00:00:1753461096.899897  227620 cuda_dnn.cc:529] Loaded cuDNN version 90501
  0% [...........................] 1/3000 in 56:372025-07-25 13:31:37.370843: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
  0% [...........................] 3/3000 in 33:112025-07-25 13:31:38.410209: I tensorflow/

Skipped 40 trial(s) due to ResourceExhaustedError.
Skipped 0 trial(s) due to InternalError.
Skipped 0 trial(s) due to UnavailableError.
Skipped 0 trial(s) due to cuDNN scratch space error.
